In [1]:
# ============================================================
# COMMENT REPLIES NLP + K-MEANS CLUSTERING
# Dataset: comment_replies.csv
# ============================================================

# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

# Run this cell only if the libraries are not already installed.
# Uncomment the following line:

# !pip install pandas numpy matplotlib seaborn scikit-learn nltk


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import warnings

warnings.filterwarnings("ignore")

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD


# ============================================================
# 3. DOWNLOAD NLTK DATA
# ============================================================

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")


# ============================================================
# 4. LOAD DATASET
# ============================================================

file_name = "comment_replies.csv"

df = pd.read_csv(file_name)

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# 5. DATASET INFORMATION
# ============================================================

print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


# ============================================================
# 6. AUTOMATICALLY FIND TEXT COLUMN
# ============================================================

# Possible names for a comment/text column
possible_text_columns = [
    "comment",
    "comments",
    "comment_text",
    "comment_texts",
    "reply",
    "replies",
    "text",
    "content",
    "message",
    "body",
    "review"
]

# Convert column names to lowercase for comparison
lower_columns = {
    column.lower().strip(): column
    for column in df.columns
}

text_column = None

for possible_column in possible_text_columns:
    if possible_column in lower_columns:
        text_column = lower_columns[possible_column]
        break


# If no common name is found, find the column containing the most text
if text_column is None:

    object_columns = df.select_dtypes(
        include=["object"]
    ).columns.tolist()

    if len(object_columns) == 0:
        raise ValueError(
            "No text column was found in the dataset."
        )

    average_lengths = {}

    for column in object_columns:
        average_lengths[column] = (
            df[column]
            .fillna("")
            .astype(str)
            .str.len()
            .mean()
        )

    text_column = max(
        average_lengths,
        key=average_lengths.get
    )


print("\nSelected text column:", text_column)


# ============================================================
# 7. REMOVE MISSING COMMENTS
# ============================================================

df = df.dropna(
    subset=[text_column]
).copy()

df[text_column] = df[text_column].astype(str)

print("\nRows after removing missing comments:", len(df))


# ============================================================
# 8. REMOVE DUPLICATE COMMENTS
# ============================================================

before_duplicates = len(df)

df = df.drop_duplicates(
    subset=[text_column]
).copy()

after_duplicates = len(df)

print(
    "Duplicate comments removed:",
    before_duplicates - after_duplicates
)

print(
    "Rows remaining:",
    len(df)
)


# ============================================================
# 9. NLP PREPROCESSING
# ============================================================

stop_words = set(
    stopwords.words("english")
)

lemmatizer = WordNetLemmatizer()


def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove mentions
    text = re.sub(
        r"@\w+",
        " ",
        text
    )

    # Remove hashtag symbol
    text = re.sub(
        r"#",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Keep only alphabetic characters
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords
    words = [
        word
        for word in words
        if word not in stop_words
    ]

    # Remove very short words
    words = [
        word
        for word in words
        if len(word) > 2
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)


# Apply preprocessing
df["clean_comment"] = df[text_column].apply(
    clean_text
)


# ============================================================
# 10. DISPLAY BEFORE/AFTER NLP
# ============================================================

print("=" * 70)
print("NLP PREPROCESSING EXAMPLES")
print("=" * 70)

display(
    df[
        [text_column, "clean_comment"]
    ].head(10)
)


# ============================================================
# 11. REMOVE EMPTY TEXT
# ============================================================

df = df[
    df["clean_comment"].str.strip() != ""
].copy()

df = df.reset_index(drop=True)

print(
    "\nComments remaining after NLP cleaning:",
    len(df)
)


# ============================================================
# 12. TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(
    df["clean_comment"]
)

print("=" * 70)
print("TF-IDF")
print("=" * 70)

print("TF-IDF matrix shape:", X.shape)

print(
    "Number of documents:",
    X.shape[0]
)

print(
    "Number of features:",
    X.shape[1]
)


# ============================================================
# 13. FIND OPTIMAL NUMBER OF CLUSTERS
# ============================================================

# We will test k from 2 to 10.
# If the dataset is very small, reduce the maximum k.

max_k = min(
    10,
    len(df) - 1
)

k_values = range(
    2,
    max_k + 1
)

inertia_values = []
silhouette_values = []


print("=" * 70)
print("TESTING DIFFERENT NUMBERS OF CLUSTERS")
print("=" * 70)

for k in k_values:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X)

    inertia = kmeans.inertia_

    silhouette = silhouette_score(
        X,
        labels
    )

    inertia_values.append(inertia)

    silhouette_values.append(
        silhouette
    )

    print(
        f"k = {k:2d} | "
        f"Inertia = {inertia:.4f} | "
        f"Silhouette Score = {silhouette:.4f}"
    )


# ============================================================
# 14. FIND BEST K USING SILHOUETTE SCORE
# ============================================================

best_k = list(k_values)[
    np.argmax(silhouette_values)
]

best_silhouette = max(
    silhouette_values
)

print("\n" + "=" * 70)
print("BEST NUMBER OF CLUSTERS")
print("=" * 70)

print("Best k:", best_k)

print(
    "Best silhouette score:",
    round(best_silhouette, 4)
)


# ============================================================
# 15. ELBOW METHOD GRAPH
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    list(k_values),
    inertia_values,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (k)"
)

plt.ylabel(
    "Inertia"
)

plt.title(
    "Elbow Method for K-Means"
)

plt.xticks(
    list(k_values)
)

plt.grid(
    True,
    alpha=0.3
)

plt.show()


# ============================================================
# 16. SILHOUETTE SCORE GRAPH
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    list(k_values),
    silhouette_values,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (k)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "Silhouette Score for Different K Values"
)

plt.xticks(
    list(k_values)
)

plt.grid(
    True,
    alpha=0.3
)

plt.show()


# ============================================================
# 17. TRAIN FINAL K-MEANS MODEL
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X)

df["cluster"] = cluster_labels


# ============================================================
# 18. CLUSTER COUNTS
# ============================================================

print("=" * 70)
print("NUMBER OF COMMENTS PER CLUSTER")
print("=" * 70)

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

display(
    cluster_counts.to_frame(
        name="Number of Comments"
    )
)


# ============================================================
# 19. CLUSTER DISTRIBUTION GRAPH
# ============================================================

plt.figure(
    figsize=(10, 6)
)

sns.countplot(
    data=df,
    x="cluster"
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Comments"
)

plt.title(
    "Number of Comments in Each Cluster"
)

plt.show()


# ============================================================
# 20. GET TOP WORDS FOR EACH CLUSTER
# ============================================================

terms = vectorizer.get_feature_names_out()

cluster_centers = kmeans.cluster_centers_

print("=" * 70)
print("TOP WORDS IN EACH CLUSTER")
print("=" * 70)


cluster_keywords = {}

for cluster_number in range(best_k):

    # Get indices of highest TF-IDF values
    top_indices = cluster_centers[
        cluster_number
    ].argsort()[::-1][:20]

    top_words = [
        terms[index]
        for index in top_indices
    ]

    cluster_keywords[
        cluster_number
    ] = top_words

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(top_words)
    )


# ============================================================
# 21. CREATE CLUSTER KEYWORDS DATAFRAME
# ============================================================

keyword_rows = []

for cluster_number, words in cluster_keywords.items():

    keyword_rows.append({
        "cluster": cluster_number,
        "top_keywords": ", ".join(words)
    })

cluster_keywords_df = pd.DataFrame(
    keyword_rows
)

print("\nCluster keyword summary:")

display(
    cluster_keywords_df
)


# ============================================================
# 22. SHOW SAMPLE COMMENTS FROM EACH CLUSTER
# ============================================================

print("=" * 70)
print("SAMPLE COMMENTS FROM EACH CLUSTER")
print("=" * 70)

for cluster_number in range(best_k):

    print(
        f"\n{'=' * 60}"
    )

    print(
        f"CLUSTER {cluster_number}"
    )

    print(
        f"{'=' * 60}"
    )

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    # Show up to 10 comments
    for comment in cluster_data[
        text_column
    ].head(10):

        print(
            "•",
            comment
        )


# ============================================================
# 23. REDUCE TF-IDF DIMENSIONS FOR VISUALIZATION
# ============================================================

# TruncatedSVD is preferable to converting a large sparse
# TF-IDF matrix into a dense matrix.

n_components = 2

svd = TruncatedSVD(
    n_components=n_components,
    random_state=42
)

X_2d = svd.fit_transform(X)


# ============================================================
# 24. ADD 2D COORDINATES TO DATAFRAME
# ============================================================

df["component_1"] = X_2d[:, 0]

df["component_2"] = X_2d[:, 1]


# ============================================================
# 25. VISUALIZE K-MEANS CLUSTERS
# ============================================================

plt.figure(
    figsize=(12, 8)
)

sns.scatterplot(
    data=df,
    x="component_1",
    y="component_2",
    hue="cluster",
    palette="tab10",
    s=70,
    alpha=0.75
)

plt.title(
    "K-Means Clustering of Comment Replies"
)

plt.xlabel(
    "SVD Component 1"
)

plt.ylabel(
    "SVD Component 2"
)

plt.legend(
    title="Cluster",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()

plt.show()


# ============================================================
# 26. CALCULATE FINAL SILHOUETTE SCORE
# ============================================================

final_silhouette = silhouette_score(
    X,
    df["cluster"]
)

print("=" * 70)
print("FINAL MODEL PERFORMANCE")
print("=" * 70)

print(
    "Number of clusters:",
    best_k
)

print(
    "Silhouette score:",
    round(final_silhouette, 4)
)

print(
    "Number of comments:",
    len(df)
)

print(
    "Number of TF-IDF features:",
    X.shape[1]
)


# ============================================================
# 27. CREATE CLUSTER SUMMARY
# ============================================================

cluster_summary = (
    df.groupby("cluster")
    .agg(
        number_of_comments=(
            text_column,
            "count"
        ),
        average_comment_length=(
            "clean_comment",
            lambda x: x.str.len().mean()
        )
    )
    .reset_index()
)

# Add percentage
cluster_summary["percentage"] = (
    cluster_summary["number_of_comments"]
    / len(df)
    * 100
)

# Round values
cluster_summary[
    "average_comment_length"
] = cluster_summary[
    "average_comment_length"
].round(2)

cluster_summary[
    "percentage"
] = cluster_summary[
    "percentage"
].round(2)

print("=" * 70)
print("CLUSTER SUMMARY")
print("=" * 70)

display(
    cluster_summary
)


# ============================================================
# 28. DISPLAY CLUSTER SUMMARY WITH KEYWORDS
# ============================================================

final_summary = cluster_summary.merge(
    cluster_keywords_df,
    on="cluster",
    how="left"
)

print("=" * 70)
print("FINAL CLUSTER SUMMARY")
print("=" * 70)

display(
    final_summary
)


# ============================================================
# 29. ADD CLUSTER LABELS
# ============================================================

# Create simple human-readable labels based on keywords.

cluster_labels_dict = {}

for cluster_number in range(best_k):

    words = cluster_keywords[
        cluster_number
    ]

    # Use the first 3 keywords as a basic label
    label = (
        " / ".join(words[:3])
    )

    cluster_labels_dict[
        cluster_number
    ] = label


df["cluster_label"] = (
    df["cluster"]
    .map(cluster_labels_dict)
)


# ============================================================
# 30. DISPLAY FINAL DATASET
# ============================================================

print("=" * 70)
print("FINAL DATASET")
print("=" * 70)

display(
    df[
        [
            text_column,
            "clean_comment",
            "cluster",
            "cluster_label"
        ]
    ].head(20)
)


# ============================================================
# 31. SAVE CLUSTERED DATASET
# ============================================================

output_file = (
    "comment_replies_clustered.csv"
)

# Remove visualization columns before saving
columns_to_save = [
    column
    for column in df.columns
    if column not in [
        "component_1",
        "component_2"
    ]
]

df[
    columns_to_save
].to_csv(
    output_file,
    index=False
)

print(
    "\nClustered dataset saved as:",
    output_file
)


# ============================================================
# 32. SAVE CLUSTER SUMMARY
# ============================================================

summary_file = (
    "comment_cluster_summary.csv"
)

final_summary.to_csv(
    summary_file,
    index=False
)

print(
    "Cluster summary saved as:",
    summary_file
)


# ============================================================
# 33. SAVE CLUSTERED COMMENTS BY CLUSTER
# ============================================================

for cluster_number in range(best_k):

    cluster_file = (
        f"cluster_{cluster_number}_comments.csv"
    )

    df[
        df["cluster"] == cluster_number
    ].to_csv(
        cluster_file,
        index=False
    )

print(
    "\nIndividual cluster files have also been created."
)


# ============================================================
# 34. FINAL REPORT
# ============================================================

print("\n")
print("=" * 70)
print("FINAL NLP + K-MEANS REPORT")
print("=" * 70)

print(
    f"Dataset: {file_name}"
)

print(
    f"Text column: {text_column}"
)

print(
    f"Total comments analysed: {len(df)}"
)

print(
    f"TF-IDF features: {X.shape[1]}"
)

print(
    f"Optimal number of clusters: {best_k}"
)

print(
    f"Silhouette score: {final_silhouette:.4f}"
)

print("\nCluster sizes:")

for cluster_number in range(best_k):

    count = (
        df["cluster"] == cluster_number
    ).sum()

    percentage = (
        count / len(df) * 100
    )

    print(
        f"Cluster {cluster_number}: "
        f"{count} comments "
        f"({percentage:.2f}%)"
    )

print("\nTop keywords:")

for cluster_number in range(best_k):

    print(
        f"\nCluster {cluster_number}: "
        f"{', '.join(cluster_keywords[cluster_number][:10])}"
    )

print("\n")
print("=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


ParserError: Error tokenizing data. C error: Buffer overflow caught - possible malformed input file.
